In [1]:
%pip install numpy matplotlib ipython ipykernel scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt


***
# Завдання 1

## Опис

Перевірка гіпотези однорідності: критерій пустих блоків.

Генеруємо дві незалежні вибірки:

$\bar{X} = \left( X_1, ..., X_n \right) \sim F_{\xi}(u) = 1 - e^{-u}, u \ge 0$;

$\bar{Y} = \left( Y_1, ..., Y_m \right) \sim F_{\xi}(u) = 1 - e^{-1.2 u}, u \ge 0$.

За допомогою критерію пустих блоків перевірити гіпотезу однорідності при наступних значеннях параметрів:

а) $n=500, m=1000$; b) $n=5 00 0, m=10 00 0$; с) $n=50 0 00, m=100 0 00$.

## Вимоги

$\gamma = 0.05$

$n = 5 \cdot 10^2, 5 \cdot 10^3, 5 \cdot 10^4$

$m = 10^3, 10^4, 10^5$

## Реалізація

In [3]:
def F(u: float, lambd: float) -> float:
    return 1 - np.exp(-lambd * u)

def F_s(omega: float, lambd: float) -> float:
    return -1/lambd * np.log(omega)

def generate(n: int, lambd: float):
    omega = np.random.random(n)
    X = F_s(omega, lambd)
    return X

gamma = 0.05

ns = np.array([5*10**2, 5*10**3, 5*10**4])
ms = np.array([10**3, 10**4, 10**5])

Xs = [generate(n, 1) for n in ns]
Ys = [generate(m, 1.2) for m in ms]

# Xs = [F(x, 1) for x in X]
# Ys = [F(y, 1) for y in Y]

Критерій перевірки гіпотези $H_0$ для великих значень $n$ та $m$:
1. за заданого рівня значимості $\gamma$ знаходимо $z_{\gamma}$ з рівняння $\Phi(z_{\gamma}) = \frac{1}{2}-\gamma$(Лаплас (0 - z)), 
2. обчислюємо кількість пустих блоків $\alpha_0(n, m)$- кількість напівінтервалів, утворених порядковими статистиками $\bar{X}$, що не містять жодного елемента вибірки $\bar{Y}$;
3. Визначаємо критичну область $Z_1 = \left\{ k: k > \frac{n}{1+\rho} + z_{\gamma} \cdot \frac{\rho \sqrt{n}}{(1 + \rho)^{3/2}} \right\}$; ($\rho = m/n$)
4. Якщо $k \notin Z_1$, то робимо висновок: статистичні дані не суперечать гіпотезі $H_0$. Якщо ж $k \in Z_1$, то слід прийняти альтернативну гіпотезу $H_1$.

In [4]:
def EmptyBlock(Xs, Ys, ns, ms, gamma):
    z = stats.norm.ppf(1 - gamma)
    for i, (X, Y) in enumerate(zip(Xs, Ys)):
        n = ns[i]
        m = ms[i]
        X_sorted = np.sort(X)
        bins = np.concatenate([[-np.inf], X_sorted, [np.inf]])
        counts, _ = np.histogram(Y, bins=bins)

        alpha = np.sum(counts == 0)

        rho = m / n

        opro = 1 + rho
        M = n/opro
        sigma = (rho * np.sqrt(n))/opro**(3/2)

        crit = M + z * sigma
        compare = alpha <= crit
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<=" if compare else ">"

        print(f"{status:<12} | {alpha:5} {sign:<2} {crit:10.4f} | n = {n:5} m = {m:6}")

EmptyBlock(Xs, Ys, ns, ms, gamma)

[Accept H0]  |   180 <=   180.8233 | n =   500 m =   1000
[Accept H0]  |  1641 <=  1711.4339 | n =  5000 m =  10000
[Decline H0] | 16868 >  16808.2331 | n = 50000 m = 100000


***
# Завдання 2

## Опис

Перевірка гіпотези незалежності.

Генеруємо вибірку $ \left( \bar{X}, \bar{Y} \right) = \left\{ \left( X_1, Y_1 \right), ..., \left( X_n, Y_n \right) \right\}$ за наступним правилом:

$ \left\{ X_i \right\} $ – це реалізації рівномірно розподіленої на $[0, 1]$ випадкової величини $\xi$; в якості $ \left\{ Y_i \right\} $ розглянути два варіанти:
- а) $Y_i = \xi_i \cdot \eta_i$, де $ \left\{ \eta_i \right\} $ – рівномірно розподілені на проміжку $[-1, 1]$, тобто $\left( X_i, Y_i \right) = \left( \xi_i, \xi_i \cdot \eta_i \right)$;
- b) $Y_i = \xi_i + \eta_i$, де $ \left\{ \eta_i \right\} $ – рівномірно розподілені на проміжку $[-1, 1]$, тобто $\left( X_i, Y_i \right) = \left( \xi_i, \xi_i + \eta_i \right)$;

А. Критерій Спірмена.
Перевірити гіпотезу незалежності за допомогою критерія Спірмена при наступних значеннях параметра $n$: а) $n=500$; b) $n=500 0$; с) $n=500 00$.

В. Критерій Кендалла.
Перевірити гіпотезу незалежності за допомогою критерія Кендалла при наступних значеннях параметра $n$: а) $n=500$; b) $n=500 0$; с) $n=500 00$.

## Вимоги

$\gamma = 0.05$

$n = 5 \cdot 10^2, 5 \cdot 10^3, 5 \cdot 10^4$

Критерій Спірмена та Кендалла

## Реалізація

In [5]:
def generate(n: int, sign: str):
    X = np.random.uniform(0, 1, n)
    theta = np.random.uniform(-1, 1, n)
    operations = {"*": X * theta, "+": X + theta}
    Y = operations[sign]
    return np.array([X, Y])

gamma = 0.05

ns = np.array([5*10**2, 5*10**3, 5*10**4])

XYsa = [generate(n, "*") for n in ns]
XYsb = [generate(n, "+") for n in ns]


Критерій Спірмена. Критерій перевірки гіпотези $H_0$:  якщо $\left\lvert \rho_n\left( \bar{X}, \bar{Y} \right) \right\rvert < \frac{z_{\gamma}}{\sqrt{n}}$, де $z_{\gamma}: \Phi(z_{\gamma}) = \frac{1 - \gamma}{2} $ (Лаплас (0 - z)), то робимо висновок: статистичні дані не суперечать гіпотезі $H_0$.

\begin{equation}
    \rho_n\left( \bar{X}, \bar{Y} \right) = 1 - \frac{6}{n(n^2-1)} \sum\limits_{i=1}^{n}{(R_i - S_i)^2} = 1 - \frac{6}{n(n^2-1)} \sum\limits_{i=1}^{n}{(i - V_i)^2}
\end{equation}
$(R_i, S_i)$ - ранги (номери місця)

Критерій Кендалла. Критерій перевірки гіпотези $H_0$: якщо $\left\lvert \tau_n\left( \bar{X}, \bar{Y} \right) \right\rvert < \frac{2 z_{\gamma}}{3 \sqrt{n}}$, де $z_{\gamma}: \Phi(z_{\gamma}) = \frac{1 - \gamma}{2} $ (Лаплас (0 - z)), то робимо висновок: статистичні дані не суперечать гіпотезі $H_0$.

Зауваження. Статистику Кендала  рекомендується обчислювати за формулою:
\begin{equation}
    \tau_n\left( \bar{X}, \bar{Y} \right) = \frac{4 N}{n(n-1)}-1,
\end{equation}
де $N$ – це кількість тих пар індексів $(i, j), i < j$, для яких $V_i < V_j$.

In [6]:
def Spearman(XYs, ns, gamma, text=""):
    z = stats.norm.ppf(1 - gamma/2)
    for i, (X, Y) in enumerate(XYs):
        n = ns[i]
        R = np.argsort(np.argsort(X)) + 1
        S = np.argsort(np.argsort(Y)) + 1

        rho = 1 - 6 / (n * (n**2 - 1)) * np.sum((R-S)**2)

        crit = z/np.sqrt(n)
        compare = np.abs(rho) < crit
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<" if compare else ">="

        print(f"{status:<12} | {np.abs(rho):6.4f} {sign:<2} {crit:6.4f} | {text} n = {n:5}")

Spearman(XYsa, ns, gamma, "a)")
Spearman(XYsb, ns, gamma, "b)")

[Accept H0]  | 0.0022 <  0.0877 | a) n =   500
[Accept H0]  | 0.0086 <  0.0277 | a) n =  5000
[Accept H0]  | 0.0038 <  0.0088 | a) n = 50000
[Decline H0] | 0.4388 >= 0.0877 | b) n =   500
[Decline H0] | 0.4384 >= 0.0277 | b) n =  5000
[Decline H0] | 0.4303 >= 0.0088 | b) n = 50000


In [7]:
def Kendel(XYs, ns, gamma, text=""):
    z = stats.norm.ppf(1 - gamma/2)
    for i, (X, Y) in enumerate(XYs):
        n = ns[i]
        order = np.argsort(X)
        V = Y[order]

        N = 0
        for j in range(n-1):
            N += np.sum(V[j] < V[j+1:])
        
        tau = (4 * N)/(n*(n-1))-1

        crit = (2*z)/(3*np.sqrt(n))
        compare = np.abs(tau) < crit
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<" if compare else ">="

        print(f"{status:<12} | {np.abs(tau):6.4f} {sign:<2} {crit:6.4f} | {text} n = {n:5}")

Kendel(XYsa, ns, gamma, "a)")
Kendel(XYsb, ns, gamma, "b)")

[Accept H0]  | 0.0139 <  0.0584 | a) n =   500
[Accept H0]  | 0.0054 <  0.0185 | a) n =  5000
[Accept H0]  | 0.0018 <  0.0058 | a) n = 50000
[Decline H0] | 0.2994 >= 0.0584 | b) n =   500
[Decline H0] | 0.3007 >= 0.0185 | b) n =  5000
[Decline H0] | 0.2955 >= 0.0058 | b) n = 50000


***
# Завдання 3

## Опис

Перевірка гіпотези випадковості.

Припустимо, що вибірка $\bar{X} = \left( X_1, ..., X_n \right)$ утворюється за наступним правилом:

$X_i = \left( \xi_1 + ... + \xi_i \right) / i, i = 1, ..., n$, де $ \left\{ \xi_i \right\} $ – це послідовність незалежних рівномірно розподілених на $[-1, 1]$ випадкових величин.

Перевірити гіпотезу випадковості за допомогою критерію, що ґрунтується на обчисленні кількості інверсій при наступних значеннях параметра $n$: а) $n=500$; b) $n=500 0$; с) $n=500 00$.

## Вимоги

$\gamma = 0.05$

$n = 5 \cdot 10^2, 5 \cdot 10^3, 5 \cdot 10^4$

## Реалізація

In [8]:
def generate(n: int):
    xi = np.random.uniform(-1, 1, n)
    X = np.cumsum(xi)/np.arange(1, n+1)
    return X

gamma = 0.05

ns = np.array([5*10**2, 5*10**3, 5*10**4])

Xs = [generate(n) for n in ns]

Критерій перевірки гіпотези $H_0$ для великих значень $n$:
1. за заданого рівня значимості $\gamma$ знаходимо $z_{\gamma}$ з рівняння $\Phi(z_{\gamma}) = \frac{1-\gamma}{2}$(Лаплас (0 - z)), 
2. обчислюємо кількість інверсій $k = S_n(\bar{X})$ у вибірці $\bar{X}$ ($i > j, X_i > X_j$);
3. якщо $\frac{6}{n \sqrt{n}} \left\lvert k - \frac{n (n - 1)}{4} \right\rvert > z_{\gamma} $, то приймаємо гіпотезу $H_1$ тобто робимо висновок, що гіпотеза $H_0$ суперечить статистичним даним.
4. У протилежному випадку гіпотеза про незалежність і однаковий розподіл спостережень не суперечить статистичним даним.

Ймовірність помилково відкинути гіпотезу $H_0$, що насправді має місце, не перевищує $\gamma$ (похибка першого роду). Це правило можна використовувати вже при $n>10$.

In [9]:
def Inversia(Xs, ns, gamma, text=""):
    z = stats.norm.ppf(1 - gamma/2)
    for i, X in enumerate(Xs):
        n = ns[i]
        k = 0
        for j in range(n-1):
            # k += len(np.where(X[j+1:] < X[j])[0])
            k += np.sum(X[j+1:] < X[j])

        abso = np.abs(k - (n*(n-1))/4)

        crit = z*n*np.sqrt(n)/6
        compare = abso <= crit
        status = "[Accept H0]" if compare else "[Decline H0]"
        sign = "<=" if compare else ">"

        print(f"{status:<12} | {np.abs(abso):11} {sign:<2} {crit:12.4f} | {text} n = {n:5}")

Inversia(Xs, ns, gamma)

[Decline H0] |      9981.0 >     3652.1773 |  n =   500
[Decline H0] |   4441581.0 >   115491.9854 |  n =  5000
[Decline H0] | 115278258.0 >  3652177.2524 |  n = 50000
